In [3]:
import re
import pandas as pd


def parse_results(path="results.txt"):
    with open(path) as f:
        text = f.read()

    blocks = re.split(r"={40,}", text)

    rows = []
    for i, block in enumerate(blocks):
        header = re.search(
            r"(\w+(?:_\w+)*)\s+(NC|LP)\s*\|\s*(\w+)\s+encoder\s*\+\s*(\w+)\s+decoder\s*\|\s*seed=(\d+)",
            block,
        )
        if not header:
            continue

        dataset, task, encoder, decoder, seed = header.groups()

        # Search this block + next for the metrics (they follow the header)
        search = block + (blocks[i + 1] if i + 1 < len(blocks) else "")

        lr = re.search(r"Learning rate:\s*([\d.]+)", search)
        curv = re.search(r"Curvature:\s*(\w+)", search)
        time_ = re.search(r"Total time elapsed:\s*([\d.]+)s", search)
        params = re.search(r"Total number of parameters:\s*([\d.]+)", search)

        # Test metrics
        test_line = re.search(r"Test set results:\s*(.*)", search)
        test_metrics = {}
        if test_line:
            for m in re.finditer(r"(test_\w+):\s*([\d.]+)", test_line.group(1)):
                test_metrics[m.group(1)] = float(m.group(2))

        # Val metrics
        val_line = re.search(r"Val set results:\s*(.*)", search)
        val_metrics = {}
        if val_line:
            for m in re.finditer(r"(val_\w+):\s*([\d.]+)", val_line.group(1)):
                val_metrics[m.group(1)] = float(m.group(2))

        row = {
            "dataset": dataset,
            "task": task,
            "encoder": encoder,
            "decoder": decoder,
            "seed": int(seed),
            "lr": float(lr.group(1)) if lr else None,
            "curvature": curv.group(1) if curv else None,
            "time_s": float(time_.group(1)) if time_ else None,
            "params": float(params.group(1)) if params else None,
            **val_metrics,
            **test_metrics,
        }
        rows.append(row)

    return pd.DataFrame(rows)

In [16]:
a = parse_results()

In [17]:
a['dataset']

0      disease_nc
1      disease_nc
2      disease_nc
3      disease_nc
4      disease_nc
          ...    
499       airport
500       airport
501       airport
502       airport
503       airport
Name: dataset, Length: 504, dtype: object

In [19]:
summary = (
    a.groupby(["dataset", "task", "encoder", "lr", "curvature"])
    .agg(
        n=("seed", "count"),
        test_acc_mean=("test_acc", "mean"),
        test_acc_std=("test_acc", "std"),
        test_f1_mean=("test_f1", "mean"),
        test_f1_std=("test_f1", "std"),
        test_roc_mean=("test_roc", "mean"),
        test_roc_std=("test_roc", "std"),
        test_ap_mean=("test_ap", "mean"),
        test_ap_std=("test_ap", "std"),
    )
    .reset_index()
)

# NC rows: show acc/f1, blank out roc/ap
# LP rows: show roc/ap, blank out acc/f1
nc = summary[summary["task"] == "NC"].drop(columns=["test_roc_mean", "test_roc_std",
"test_ap_mean", "test_ap_std"])
lp = summary[summary["task"] == "LP"].drop(columns=["test_acc_mean", "test_acc_std",
"test_f1_mean", "test_f1_std"])

print("=== Node Classification ===")
print(nc.to_string(index=False))
print("\n=== Link Prediction ===")
print(lp.to_string(index=False))

=== Node Classification ===
   dataset task  encoder   lr curvature  n  test_acc_mean  test_acc_std  test_f1_mean  test_f1_std
   airport   NC     chen 0.01     fixed  7       0.697371      0.031471      0.697371     0.031471
   airport   NC     chen 0.01 learnable  7       0.672857      0.042281      0.672857     0.042281
   airport   NC     chen 0.05     fixed  7       0.724643      0.052530      0.724643     0.052530
   airport   NC     chen 0.05 learnable  7       0.744271      0.019296      0.744271     0.019296
   airport   NC     chen 0.10     fixed  7       0.767457      0.012546      0.767457     0.012546
   airport   NC     chen 0.10 learnable  7       0.751086      0.037080      0.751086     0.037080
   airport   NC     ours 0.01     fixed  7       0.755971      0.031090      0.755971     0.031090
   airport   NC     ours 0.01 learnable  7       0.725729      0.038142      0.725729     0.038142
   airport   NC     ours 0.05     fixed  7       0.763914      0.027273      0.76